# Detecting AI-Generated Images with HOG + LBP Features and an SVM

**Goal:** sort 32×32 images into two classes:

| Label | Class | Meaning |
|---|---|---|
| `0` | **REAL** | a real photograph |
| `1` | **FAKE** | an AI-generated image |

**Pipeline**

```
image (32x32 RGB) ──► grayscale ──┬──► HOG descriptor (shape / edge structure)  ──┐
                                  └──► LBP histograms (micro-texture)              ──┴──► concat ──► StandardScaler ──► SVM ──► REAL / FAKE
```

Why these features?
* **HOG (Histogram of Oriented Gradients)** records how edges and gradients are oriented in each small region of the image, which captures *shape and structure*.
* **LBP (Local Binary Patterns)** compares each pixel with its neighbours and records the resulting binary pattern, which captures fine *texture*. Generative models often leave small texture artifacts (overly smooth areas, repeated patterns), so LBP is well suited to spotting them.
* **SVM (Support Vector Machine)** finds the maximum-margin boundary between the two classes. With an RBF kernel it can learn non-linear boundaries in the combined HOG+LBP feature space.

The notebook has these steps:
1. Setup & imports
2. Configuration
3. Load the dataset (file paths + labels)
4. Explore and visualise the data
5. Feature extraction (HOG, LBP, combined), with visualisations
6. Extract features for the whole dataset (parallel + cached)
7. Feature scaling
8. Hyperparameter tuning (GridSearchCV)
9. Train the final SVM
10. Evaluation (metrics, confusion matrix, ROC curve, ablation, error analysis)
11. Save the model

## 1. Setup & imports

Run the cell below **once** to install the required libraries into the active environment (`%pip` installs into the kernel's own Python). You can comment it out afterwards.

In [ ]:
%pip install -q numpy scikit-learn scikit-image matplotlib pillow tqdm joblib ipywidgets

Now import everything the notebook uses:

* `numpy` – arrays and numerical work
* `PIL.Image` – reading the `.jpg` files
* `skimage.feature.hog` / `local_binary_pattern` – the two feature descriptors
* `sklearn` – scaling, the SVM, cross-validation/grid search and evaluation metrics
* `joblib` – runs feature extraction in parallel on all CPU cores and saves/loads the model
* `matplotlib` – plots

In [ ]:
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from joblib import Parallel, delayed, dump, load

from skimage.feature import hog, local_binary_pattern

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score,
)

plt.rcParams["figure.dpi"] = 100
print("Imports OK")

## 2. Configuration

All tunable settings are in one place:

* **Paths** – the dataset is expected at `dataset/train/{REAL,FAKE}` and `dataset/test/{REAL,FAKE}`, next to this notebook.
* **Class mapping** – `REAL → 0`, `FAKE → 1`. FAKE is the *positive* class, because detecting AI images is the goal.
* **HOG parameters** – on a 32×32 image, 8×8-pixel cells give a 4×4 grid of cells. Grouping 2×2 cells into blocks gives 3×3 = 9 overlapping blocks. Each block holds 2·2·9 = 36 values (9 orientation bins per cell), so **9 × 36 = 324 HOG features**.
* **LBP parameters** – `P=8` neighbours on a circle of radius `R=1`, using the `"uniform"` method. This keeps only rotation-invariant "uniform" patterns, giving `P+2 = 10` possible codes. To keep *where* each texture occurs, the LBP image is split into a 4×4 grid of 8×8 regions and a 10-bin histogram is computed for each region, so **16 × 10 = 160 LBP features**.
* **SVM training size** – training time for a kernel SVM grows roughly with the *square* of the number of samples. 100,000 images would take about an hour or more. By default the final model is trained on a stratified subset of `N_TRAIN` images. Set `N_TRAIN = None` to use all training data. The **test set is always the full 20,000 images.**
* **Grid search size** – hyperparameters are tuned on a smaller subset (`N_GRID`) with cross-validation, to save time.

In [ ]:
# ---- Paths ----
DATA_DIR  = Path("/kaggle/input/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images")
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR  = DATA_DIR / "test"
CACHE_DIR = Path("features_cache")      # extracted features are cached here
MODEL_DIR = Path("models")
CACHE_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

# ---- Classes ----
CLASSES = {"REAL": 0, "FAKE": 1}
CLASS_NAMES = ["REAL", "FAKE"]          # index == label

# ---- Image ----
IMG_SIZE = 32

# ---- HOG ----
HOG_PARAMS = dict(
    orientations=9,
    pixels_per_cell=(8, 8),
    cells_per_block=(2, 2),
    block_norm="L2-Hys",
)

# ---- LBP ----
LBP_P, LBP_R = 8, 1
LBP_METHOD   = "uniform"
LBP_BINS     = LBP_P + 2                # number of uniform codes
LBP_GRID     = 4                        # 4x4 spatial grid of regions

# ---- Training sizes ----
N_TRAIN = 30_000     # images used to train the final SVM (None = all 100k)
N_GRID  = 10_000     # images used for hyperparameter search
RANDOM_STATE = 42
N_JOBS = -1          # use all CPU cores

## 3. Load the dataset (file paths + labels)

`load_image_paths()` walks the `REAL` and `FAKE` sub-folders of a split and returns two arrays: the image file paths and their integer labels. Only paths are loaded here. The pixels are read later, during feature extraction, so the 120,000 images never need to sit in memory at the same time.

In [ ]:
def load_image_paths(split_dir):
    paths, labels = [], []
    for class_name, label in CLASSES.items():
        files = sorted((split_dir / class_name).glob("*.*"))
        paths.extend(str(f) for f in files)
        labels.extend([label] * len(files))
    return np.array(paths), np.array(labels)

train_paths, y_train_all = load_image_paths(TRAIN_DIR)
test_paths,  y_test      = load_image_paths(TEST_DIR)

print(f"Train images: {len(train_paths):,}")
print(f"Test images : {len(test_paths):,}")

## 4. Explore & visualise the data

### 4.1 Class balance
First check how many images each class has. A balanced dataset means plain accuracy is a meaningful metric, and the SVM needs no class weighting.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
for ax, (name, y) in zip(axes, [("Train", y_train_all), ("Test", y_test)]):
    counts = np.bincount(y)
    bars = ax.bar(CLASS_NAMES, counts, color=["tab:blue", "tab:orange"])
    ax.bar_label(bars, labels=[f"{c:,}" for c in counts])
    ax.set_title(f"{name} set class distribution")
    ax.set_ylabel("Number of images")
plt.tight_layout()
plt.show()

### 4.2 Sample images
Show some random images from each class, and confirm they really are 32×32. At this resolution the two classes are hard to tell apart by eye, which is why hand-crafted texture and gradient features are worth trying.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
n_show = 8
fig, axes = plt.subplots(2, n_show, figsize=(n_show * 1.5, 3.6))
for row, (name, label) in enumerate(CLASSES.items()):
    idx = rng.choice(np.where(y_train_all == label)[0], n_show, replace=False)
    for col, i in enumerate(idx):
        axes[row, col].imshow(Image.open(train_paths[i]))
        axes[row, col].axis("off")
    axes[row, 0].set_title(name, loc="left", fontsize=11, fontweight="bold")
plt.suptitle("Random training samples")
plt.tight_layout()
plt.show()

sample = Image.open(train_paths[0])
print("Example image size:", sample.size, "mode:", sample.mode)

## 5. Feature extraction

Both descriptors work on a single intensity channel, so every image is first converted to **grayscale** (`PIL .convert("L")`) and resized to 32×32 as a safety check. The dataset is already 32×32, so the resize does nothing here.

### 5.1 HOG feature extractor
`extract_hog()`:
1. computes the gradient (magnitude + direction) at every pixel,
2. builds a 9-bin histogram of gradient directions inside each 8×8 cell, weighted by magnitude,
3. normalises the histograms over overlapping 2×2-cell blocks (`L2-Hys`), which makes the result robust to lighting and contrast changes,
4. flattens everything into one 324-value vector.

With `visualize=True` it also returns a picture of the dominant gradient directions, used in 5.4.

In [ ]:
def load_gray(path):
    img = Image.open(path).convert("L")
    if img.size != (IMG_SIZE, IMG_SIZE):
        img = img.resize((IMG_SIZE, IMG_SIZE))
    return np.asarray(img, dtype=np.uint8)


def extract_hog(gray, visualize=False):
    return hog(gray, feature_vector=True, visualize=visualize, **HOG_PARAMS)

### 5.2 LBP feature extractor
`extract_lbp()`:
1. computes the **LBP code** of every pixel: each of the 8 neighbours on a radius-1 circle is compared with the centre (1 if ≥ centre, else 0), and the resulting 8-bit pattern is mapped to one of 10 *uniform* codes,
2. splits the 32×32 LBP map into a **4×4 grid** of 8×8 regions,
3. computes a normalised 10-bin histogram of codes in each region,
4. concatenates the 16 histograms into a 160-value vector.

A single global histogram would lose all spatial information. The grid keeps a rough idea of *where* each texture appears, for example smooth backgrounds versus a detailed centre.

In [ ]:
def extract_lbp(gray, return_map=False):
    lbp = local_binary_pattern(gray, LBP_P, LBP_R, method=LBP_METHOD)
    step = IMG_SIZE // LBP_GRID
    hists = []
    for i in range(0, IMG_SIZE, step):
        for j in range(0, IMG_SIZE, step):
            region = lbp[i:i + step, j:j + step]
            h, _ = np.histogram(region, bins=LBP_BINS, range=(0, LBP_BINS))
            hists.append(h / region.size)          # normalise -> frequencies
    feats = np.concatenate(hists)
    return (feats, lbp) if return_map else feats

### 5.3 Combined HOG + LBP extractor
`extract_features()` builds the final descriptor for one image by joining (concatenating) the two vectors: **324 (HOG) + 160 (LBP) = 484 features**. The result is stored as `float32` to halve memory use.

The cell then runs it on one image to confirm the dimensions.

In [ ]:
def extract_features(path):
    gray = load_gray(path)
    return np.concatenate([extract_hog(gray), extract_lbp(gray)]).astype(np.float32)

_g = load_gray(train_paths[0])
N_HOG = extract_hog(_g).shape[0]
print("HOG length     :", N_HOG)
print("LBP length     :", extract_lbp(_g).shape[0])
print("Combined length:", extract_features(train_paths[0]).shape[0])

### 5.4 Visualise the features
For one REAL and one FAKE image, show:
* the grayscale input,
* the **HOG visualisation**, where each cell shows its strongest gradient orientations,
* the **LBP code map**, where each pixel's colour is its texture pattern,
* the **overall LBP histogram**, which shows how often each texture pattern occurs.

Comparing the histograms gives an early hint of the texture differences between real photos and generated images.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 6))
for row, (name, label) in enumerate(CLASSES.items()):
    path = train_paths[rng.choice(np.where(y_train_all == label)[0])]
    gray = load_gray(path)
    _, hog_img = extract_hog(gray, visualize=True)
    _, lbp_map = extract_lbp(gray, return_map=True)

    axes[row, 0].imshow(gray, cmap="gray");             axes[row, 0].set_title(f"{name}: grayscale")
    axes[row, 1].imshow(hog_img, cmap="magma");         axes[row, 1].set_title("HOG")
    axes[row, 2].imshow(lbp_map, cmap="nipy_spectral"); axes[row, 2].set_title("LBP codes")
    axes[row, 3].bar(range(LBP_BINS),
                     np.bincount(lbp_map.astype(int).ravel(), minlength=LBP_BINS) / lbp_map.size)
    axes[row, 3].set_title("LBP histogram"); axes[row, 3].set_xlabel("uniform code")
    for ax in axes[row, :3]:
        ax.axis("off")
plt.tight_layout()
plt.show()

## 6. Extract features for the whole dataset

Running the extractor on 120,000 images one at a time would take about 25 minutes. `build_feature_matrix()`:
* runs `extract_features` in **parallel** on all CPU cores with `joblib`,
* **caches** the resulting matrix to `features_cache/*.npz`, so re-running the notebook loads features in seconds,
* checks that the cached labels match the current file list, so a stale cache is never used silently.

**If you change the HOG/LBP parameters, delete the `features_cache` folder** so the features are recomputed.

The result is a matrix `X` of shape *(n_images, 484)*, where each row is one image's descriptor.

In [ ]:
def build_feature_matrix(paths, labels, cache_name):
    cache_file = CACHE_DIR / f"{cache_name}.npz"
    if cache_file.exists():
        data = np.load(cache_file)
        if np.array_equal(data["y"], labels):
            print(f"Loaded cached features: {cache_file}")
            return data["X"], data["y"]
        print("Cache does not match dataset - recomputing.")

    t0 = time.time()
    feats = Parallel(n_jobs=N_JOBS, batch_size=256)(
        delayed(extract_features)(p) for p in tqdm(paths, desc=f"Extracting {cache_name}")
    )
    X = np.vstack(feats)
    np.savez_compressed(cache_file, X=X, y=labels)
    print(f"Extracted {X.shape} in {time.time() - t0:.1f}s -> cached to {cache_file}")
    return X, labels

X_train_all, y_train_all = build_feature_matrix(train_paths, y_train_all, "train_hog_lbp")
X_test,      y_test      = build_feature_matrix(test_paths,  y_test,      "test_hog_lbp")

print("X_train_all:", X_train_all.shape, " X_test:", X_test.shape)

### 6.1 Select the training subset
As explained in the configuration, a **stratified** random subset of `N_TRAIN` training images is taken. Stratified means it keeps the 50/50 REAL/FAKE balance. A smaller subset of `N_GRID` images is also taken for hyperparameter search. The test set is not touched.

In [ ]:
def stratified_subset(X, y, n, seed=RANDOM_STATE):
    if n is None or n >= len(y):
        return X, y
    X_sub, _, y_sub, _ = train_test_split(X, y, train_size=n, stratify=y, random_state=seed)
    return X_sub, y_sub

X_train, y_train = stratified_subset(X_train_all, y_train_all, N_TRAIN)
X_grid,  y_grid  = stratified_subset(X_train_all, y_train_all, N_GRID)

print(f"Final-training subset : {X_train.shape}, class counts = {np.bincount(y_train)}")
print(f"Grid-search subset    : {X_grid.shape}, class counts = {np.bincount(y_grid)}")

## 7. Feature scaling

SVMs are **sensitive to feature scale**. The RBF kernel uses Euclidean distances, so features with large ranges would dominate. HOG values (block-normalised, up to about 0.2–0.3) and LBP frequencies (0–1) sit on different scales, so `StandardScaler` rescales every feature to mean 0 and standard deviation 1.

To avoid **data leakage**, the scaler is never fitted by hand on all the data. Instead it goes inside an sklearn `Pipeline(scaler → SVM)`, so during cross-validation and final training it only learns its mean and standard deviation from training data. The cell below just shows what scaling does to the feature ranges. The red dashed line marks the boundary between HOG and LBP features.

In [ ]:
X_demo = StandardScaler().fit_transform(X_train)

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].plot(X_train.mean(0), lw=0.8, label="mean"); axes[0].plot(X_train.std(0), lw=0.8, label="std")
axes[0].set_title("Raw features"); axes[0].legend()
axes[1].plot(X_demo.mean(0), lw=0.8, label="mean"); axes[1].plot(X_demo.std(0), lw=0.8, label="std")
axes[1].set_title("After StandardScaler"); axes[1].legend()
for ax in axes:
    ax.axvline(N_HOG, color="red", ls="--", lw=0.8)
    ax.set_xlabel("feature index  (HOG | LBP)")
plt.tight_layout()
plt.show()
del X_demo

## 8. Hyperparameter tuning with GridSearchCV

An RBF-kernel SVM has two key hyperparameters:
* **`C`** – the regularisation strength. A small `C` allows a wider margin with more training mistakes (simpler model). A large `C` tries to classify every training point correctly (risk of overfitting).
* **`gamma`** – the reach of each support vector. A small gamma gives smooth decision boundaries, and a large gamma gives very wiggly ones. `"scale"` means `1 / (n_features · Var(X))`, which is about 0.002 here after scaling.

`GridSearchCV` tries every combination with **3-fold stratified cross-validation** on the `N_GRID` subset and picks the one with the best mean accuracy. Each candidate is a full `scaler → SVC` pipeline, so scaling is redone inside every fold. This takes a few minutes.

In [ ]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf")),
])

param_grid = {
    "svm__C":     [1, 10, 30],
    "svm__gamma": ["scale", 0.001, 0.005],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
grid = GridSearchCV(pipe, param_grid, cv=cv, scoring="accuracy", n_jobs=N_JOBS, verbose=1)

t0 = time.time()
grid.fit(X_grid, y_grid)
print(f"Grid search finished in {time.time() - t0:.1f}s")
print("Best params :", grid.best_params_)
print(f"Best CV acc : {grid.best_score_:.4f}")

The heat-map below shows the mean CV accuracy for each (`C`, `gamma`) pair. If the best value sits on the edge of the grid, it is worth widening the grid in that direction.

In [ ]:
res = grid.cv_results_
Cs, gammas = param_grid["svm__C"], param_grid["svm__gamma"]
scores = np.zeros((len(Cs), len(gammas)))
for p, s in zip(res["params"], res["mean_test_score"]):
    scores[Cs.index(p["svm__C"]), gammas.index(p["svm__gamma"])] = s

fig, ax = plt.subplots(figsize=(5, 3.5))
im = ax.imshow(scores, cmap="viridis")
ax.set_xticks(range(len(gammas)), [str(g) for g in gammas]); ax.set_xlabel("gamma")
ax.set_yticks(range(len(Cs)), [str(c) for c in Cs]);         ax.set_ylabel("C")
for i in range(len(Cs)):
    for j in range(len(gammas)):
        ax.text(j, i, f"{scores[i, j]:.3f}", ha="center", va="center", color="w")
ax.set_title("3-fold CV accuracy")
fig.colorbar(im)
plt.tight_layout()
plt.show()

## 9. Train the final SVM

Build a fresh `scaler → SVC` pipeline with the **best hyperparameters** and train it on the larger `N_TRAIN` subset. More data generally helps an SVM, but training time grows quickly, which is why `N_TRAIN` is configurable. 30k images take about 5 minutes.

`probability=False` (the default) is kept for speed. The ROC curve below uses the SVM's `decision_function` (signed distance from the margin), which is all ROC/AUC needs.

In [ ]:
best = {k.replace("svm__", ""): v for k, v in grid.best_params_.items()}

model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", **best)),
])

t0 = time.time()
model.fit(X_train, y_train)
train_time = time.time() - t0

svm = model.named_steps["svm"]
print(f"Trained on {len(y_train):,} images in {train_time:.1f}s")
print("Hyperparameters :", best)
print("Support vectors :", svm.n_support_.sum(), "(per class:", dict(zip(CLASS_NAMES, svm.n_support_.tolist())), ")")

## 10. Evaluation

### 10.1 Predict on the test set
Predict labels for all **20,000 unseen test images**. Also record the decision scores, used for ROC/AUC. The training-subset accuracy is printed too: a large gap between training and test accuracy would point to overfitting.

In [ ]:
t0 = time.time()
y_pred  = model.predict(X_test)
y_score = model.decision_function(X_test)     # >0 -> FAKE, <0 -> REAL
print(f"Predicted {len(y_test):,} test images in {time.time() - t0:.1f}s")

train_acc = accuracy_score(y_train, model.predict(X_train))
print(f"Train accuracy (subset): {train_acc:.4f}")
print(f"Test accuracy          : {accuracy_score(y_test, y_pred):.4f}")

### 10.2 Summary metrics
* **Accuracy** – the fraction of all test images classified correctly.
* **Precision (FAKE)** – of the images predicted FAKE, how many really are FAKE (how often a "fake" alarm is right).
* **Recall (FAKE)** – of all the truly FAKE images, how many were caught.
* **F1** – the harmonic mean of precision and recall.
* **ROC-AUC** – how well the decision score ranks FAKE above REAL across *all* thresholds (1.0 = perfect, 0.5 = random).

The **classification report** gives precision, recall and F1 for *each* class.

In [ ]:
metrics = {
    "Accuracy":         accuracy_score(y_test, y_pred),
    "Precision (FAKE)": precision_score(y_test, y_pred),
    "Recall (FAKE)":    recall_score(y_test, y_pred),
    "F1 (FAKE)":        f1_score(y_test, y_pred),
    "ROC-AUC":          roc_auc_score(y_test, y_score),
}
for k, v in metrics.items():
    print(f"{k:<17}: {v:.4f}")

print("\nClassification report:\n")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4))

### 10.3 Confusion matrix
Rows are the **true** class and columns the **predicted** class. The diagonal shows correct predictions. The off-diagonal cells are the two error types:
* **REAL → FAKE** (false positive): a real photo flagged as AI-generated.
* **FAKE → REAL** (false negative): an AI image that went undetected.

The right-hand plot normalises each row, so the cells read as per-class rates.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=axes[0], cmap="Blues", values_format=",")
axes[0].set_title("Confusion matrix (counts)")
ConfusionMatrixDisplay(cm / cm.sum(1, keepdims=True), display_labels=CLASS_NAMES).plot(
    ax=axes[1], cmap="Blues", values_format=".2%")
axes[1].set_title("Confusion matrix (row-normalised)")
plt.tight_layout()
plt.show()

### 10.4 ROC curve
The ROC curve plots the **true positive rate** (FAKE caught) against the **false positive rate** (REAL wrongly flagged) as the decision threshold moves. The default SVM threshold is 0 and is marked on the curve. A curve that hugs the top-left corner, with an AUC close to 1, means good separation.

In [ ]:
fpr, tpr, thr = roc_curve(y_test, y_score)
i0 = np.argmin(np.abs(thr))                    # point closest to threshold 0

plt.figure(figsize=(5, 4.5))
plt.plot(fpr, tpr, label=f"HOG+LBP SVM (AUC = {metrics['ROC-AUC']:.3f})")
plt.plot([0, 1], [0, 1], "k--", lw=0.8, label="Random")
plt.scatter(fpr[i0], tpr[i0], c="red", zorder=3, label="threshold = 0")
plt.xlabel("False positive rate (REAL flagged as FAKE)")
plt.ylabel("True positive rate (FAKE detected)")
plt.title("ROC curve - test set")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 10.5 Contribution of each feature type (ablation)
To check that **both** descriptors help, train the same SVM (same hyperparameters, same training subset) on **HOG only** and **LBP only**, and compare against the combined model. Feature columns `0–323` are HOG and `324–483` are LBP.

To keep this quick, it uses the `N_GRID` subset for training. The combined model is retrained on that same subset, so all three comparisons are fair.

In [ ]:
feature_sets = {
    "HOG only":  slice(0, N_HOG),
    "LBP only":  slice(N_HOG, None),
    "HOG + LBP": slice(0, None),
}

ablation = {}
for name, cols in feature_sets.items():
    m = Pipeline([("scaler", StandardScaler()), ("svm", SVC(kernel="rbf", **best))])
    m.fit(X_grid[:, cols], y_grid)
    ablation[name] = accuracy_score(y_test, m.predict(X_test[:, cols]))
    print(f"{name:<10} ({X_grid[:, cols].shape[1]:>3} features): test accuracy = {ablation[name]:.4f}")

plt.figure(figsize=(5, 3))
bars = plt.bar(ablation.keys(), ablation.values(), color=["tab:green", "tab:purple", "tab:red"])
plt.bar_label(bars, fmt="%.3f")
plt.ylim(0.5, 1.0); plt.ylabel("Test accuracy")
plt.title(f"Feature ablation (trained on {len(y_grid):,} images)")
plt.tight_layout()
plt.show()

### 10.6 Error analysis – misclassified examples
Looking at *which* images the model gets wrong shows where hand-crafted features fall short. Common examples are very smooth real photos that look "too clean", or AI images with rich, natural-looking texture. Below are random misclassified test images, each with its true label, the predicted label and the decision score. A score near 0 means the model was unsure.

In [ ]:
wrong = np.where(y_pred != y_test)[0]
print(f"Misclassified: {len(wrong):,} / {len(y_test):,}")

show = rng.choice(wrong, min(16, len(wrong)), replace=False)
fig, axes = plt.subplots(2, 8, figsize=(14, 4.2))
for ax in axes.ravel():
    ax.axis("off")
for ax, i in zip(axes.ravel(), show):
    ax.imshow(Image.open(test_paths[i]))
    ax.set_title(f"T:{CLASS_NAMES[y_test[i]]} P:{CLASS_NAMES[y_pred[i]]}\nscore={y_score[i]:+.2f}", fontsize=8)
plt.suptitle("Misclassified test images (T = true, P = predicted)")
plt.tight_layout()
plt.show()

## 11. Save the trained model

Save the whole pipeline (scaler + SVM) together with the feature configuration using `joblib`. A future image then goes through **exactly** the same preprocessing. The `predict_image()` helper shows how to classify a new image file with the saved model.

In [ ]:
model_path = MODEL_DIR / "hog_lbp_svm.joblib"
dump({"model": model, "hog_params": HOG_PARAMS,
      "lbp": dict(P=LBP_P, R=LBP_R, method=LBP_METHOD, grid=LBP_GRID),
      "classes": CLASS_NAMES, "metrics": metrics}, model_path)
print("Saved to", model_path)


def predict_image(path, bundle):
    x = extract_features(path).reshape(1, -1)
    score = bundle["model"].decision_function(x)[0]
    return bundle["classes"][int(score > 0)], score

# quick check on one test image of each class
bundle = load(model_path)
for label in (0, 1):
    p = test_paths[np.where(y_test == label)[0][0]]
    pred, s = predict_image(p, bundle)
    print(f"{Path(p).parent.name}/{Path(p).name:<15} -> predicted {pred} (score {s:+.2f})")

## Summary

* Each 32×32 image was described by **324 HOG features** (edge and shape structure) plus **160 spatial LBP features** (micro-texture), for 484 in total.
* Features were standardised and classified with an **RBF-kernel SVM**, with `C` and `gamma` chosen by 3-fold cross-validated grid search.
* The model was evaluated on the full, held-out **20,000-image test set** using accuracy, precision, recall, F1, ROC-AUC and a confusion matrix. The ablation shows how much each descriptor contributes.